[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fvalenzuelag/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA1/IL1.1/4-langchain_memory.ipynb)


# 4. LangChain Memory - Gestión de Contexto Conversacional

## Objetivos de Aprendizaje
- Comprender la importancia de la memoria en conversaciones con LLMs
- Implementar diferentes tipos de memoria con LangChain
- Gestionar el contexto de conversaciones largas
- Optimizar el uso de tokens con estrategias de memoria

## ¿Por qué es Importante la Memoria?

Los LLMs son **stateless** por naturaleza: no recuerdan conversaciones anteriores. La memoria permite:
- **Contexto conversacional**: Referirse a mensajes anteriores
- **Personalización**: Recordar preferencias del usuario
- **Continuidad**: Mantener hilos de conversación coherentes
- **Experiencia natural**: Conversaciones que se sienten humanas

## Tipos de Memoria en LangChain

1. **ConversationBufferMemory**: Mantiene todo el historial
2. **ConversationSummaryMemory**: Resume conversaciones largas
3. **ConversationBufferWindowMemory**: Mantiene solo los N mensajes más recientes
4. **ConversationSummaryBufferMemory**: Combina resumen + buffer reciente

In [1]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain-openai python-dotenv


In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

# Importar bibliotecas necesarias para memoria
from langchain_openai import ChatOpenAI
# NOTA: las clases clásicas de memoria (ConversationBufferMemory,
# ConversationSummaryMemory, ConversationBufferWindowMemory) que se describen
# más abajo están DEPRECADAS en LangChain v1. Este notebook usa el reemplazo
# oficial: RunnableWithMessageHistory. Si quieres experimentar con las clásicas:
#     from langchain_classic.memory import ConversationBufferMemory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

import os

print("✓ Bibliotecas de memoria importadas correctamente")

✓ Bibliotecas de memoria importadas correctamente


In [3]:
# Configuración del modelo para memoria
try:
    llm = ChatOpenAI(
        base_url=os.getenv("LLM_BASE_URL"),
        api_key=os.getenv("LLM_API_KEY"),
        model=os.getenv("LLM_MODEL_SMALL", "llama-3.1-8b-instant"),
        temperature=0.1
    )
    
    print("✓ Modelo configurado para experimentos de memoria")
    print(f"Modelo: {llm.model_name}")
    
except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica las variables de entorno")

✓ Modelo configurado para experimentos de memoria
Modelo: ministral-8b-latest


In [4]:
# Prompt con historial + entrada del usuario
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de historiales
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver con memoria
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

<ruta-local>:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 1. ConversationBufferMemory - Memoria Completa

Esta memoria mantiene **todo** el historial de la conversación. Es la más simple pero puede consumir muchos tokens.

In [5]:
# Ejemplo básico con RunnableWithMessageHistory

# Prompt con historial + entrada
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de memorias por sesión
store = {}

def get_session_history(session_id: str):
    """Devuelve (o crea) el historial completo para la sesión."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver con RunnableWithMessageHistory
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejemplo_buffer_memory():
    print("=== CONVERSATIONBUFFERMEMORY ===")
    print("Mantiene todo el historial de conversación\n")
    
    session_id = "demo_session"

    try:
        # Primera interacción
        print("1. Primera pregunta:")
        response1 = conversation.invoke(
            {"input": "Mi nombre es Ana y soy programadora Python"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response1.content}\n")

        # Segunda interacción
        print("2. Segunda pregunta:")
        response2 = conversation.invoke(
            {"input": "¿Cuál es mi nombre y profesión?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response2.content}\n")

        # Tercera interacción
        print("3. Tercera pregunta:")
        response3 = conversation.invoke(
            {"input": "¿Qué lenguaje de programación mencioné?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response3.content}\n")

        # Mostrar historial
        print("=== CONTENIDO DE LA MEMORIA ===")
        history = store[session_id].messages
        for i, msg in enumerate(history, 1):
            print(f"{i}. {msg.type}: {msg.content}")

    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_buffer_memory()


=== CONVERSATIONBUFFERMEMORY ===
Mantiene todo el historial de conversación

1. Primera pregunta:


<ruta-local>:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Respuesta: ¡Hola Ana! 😊 Encantado de conocerte. Soy tu asistente virtual, aquí para ayudarte con todo lo relacionado con **Python** y más allá. ¿En qué puedo ayudarte hoy?

Algunas cosas en las que puedo apoyarte:
- **Código Python**: Debugging, optimización, explicaciones de algoritmos, patrones de diseño, etc.
- **Proyectos**: Ideas, estructura, librerías útiles (Pandas, NumPy, Flask, Django, TensorFlow, etc.).
- **Buenas prácticas**: PEP 8, testing (pytest), documentación (docstrings, Sphinx), etc.
- **Herramientas**: VS Code, Jupyter, Git, Docker, etc.
- **Conceptos avanzados**: Paralelismo (multiprocesamiento, asyncio), bases de datos (SQLAlchemy, SQLite), IA/ML, etc.
- **Curiosidades**: Historias de Python, curiosidades técnicas, o incluso consejos para crecer profesionalmente.

**Ejemplo de cómo podrías pedirme ayuda**:
*"Ana, tengo este error en mi código de Flask: `TypeError: 'NoneType' object is not iterable`. ¿Podrías revisarlo?"* (y me compartes el código).

**O**:
*"Necesi

Respuesta: ¡Tu nombre es **Ana** y tu profesión es **programadora en Python**! 🎉

(¡Lo recordé perfectamente gracias a tu presentación inicial! 😄). Si necesitas algo más sobre tu perfil profesional o cómo destacarte en el mundo de la programación, aquí estoy. 🚀

¿Quieres que profundicemos en algo relacionado con tu trabajo? Por ejemplo:
- **Tendencias en Python 2024** (como *typing*, *asyncio* o *librerías modernas*).
- **Cómo optimizar tu CV** para roles de *Backend*, *Data Science* o *DevOps*.
- **Proyectos para practicar** (ej: un API con FastAPI, un scraper con BeautifulSoup, etc.).

3. Tercera pregunta:


Respuesta: ¡Tú mencionaste **Python** como tu lenguaje de programación favorito (y tu profesión) al presentarte! 🐍

Si quieres, puedo ayudarte con:
- **Sintaxis avanzada** (ej: decoradores, generadores, metaclasses).
- **Librerías clave** (Pandas, NumPy, Django, Flask, TensorFlow, etc.).
- **Buenas prácticas** (PEP 8, testing, diseño limpio).
- **Proyectos prácticos** (ej: un chatbot con NLP, un scraper web, una app con FastAPI).

¿Hay algo específico de Python que quieras explorar hoy? 😊

=== CONTENIDO DE LA MEMORIA ===
1. human: Mi nombre es Ana y soy programadora Python
2. ai: ¡Hola Ana! 😊 Encantado de conocerte. Soy tu asistente virtual, aquí para ayudarte con todo lo relacionado con **Python** y más allá. ¿En qué puedo ayudarte hoy?

Algunas cosas en las que puedo apoyarte:
- **Código Python**: Debugging, optimización, explicaciones de algoritmos, patrones de diseño, etc.
- **Proyectos**: Ideas, estructura, librerías útiles (Pandas, NumPy, Flask, Django, TensorFlow, etc.).
- **Bue

## 2. ConversationBufferWindowMemory - Ventana Deslizante

Esta memoria mantiene solo los **N mensajes más recientes**, útil para controlar el uso de tokens.

In [6]:
# Prompt con historial + entrada
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de memorias por sesión
store = {}

class WindowChatMessageHistory(BaseChatMessageHistory):
    """Historial de chat que mantiene solo los últimos k intercambios."""
    
    def __init__(self, k: int = 2):
        self.k = k
        self._messages = []
    
    @property
    def messages(self):
        # Mantener solo los últimos k intercambios (k*2 mensajes: user + assistant)
        return self._messages[-(self.k * 2):]
    
    def add_message(self, message):
        self._messages.append(message)
    
    def clear(self):
        self._messages.clear()

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Devuelve el historial de ventana para la sesión."""
    if session_id not in store:
        store[session_id] = WindowChatMessageHistory(k=2)
    return store[session_id]

# Envolver con RunnableWithMessageHistory
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Ejemplo
def ejemplo_window_memory():
    print("=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===")
    print("Mantiene solo los 2 intercambios más recientes\n")
    
    session_id = "demo_window"
    inputs = [
        "Mi nombre es Carlos y tengo 30 años",
        "Trabajo como diseñador gráfico", 
        "Me gusta el café y la música jazz",
        "¿Puedes recordar mi edad?",
        "¿Cuál es mi profesión?"
    ]
    
    try:
        for i, user_input in enumerate(inputs, 1):
            print(f"{'='*20} INTERACCIÓN {i} {'='*20}")
            print(f"👤 Usuario: {user_input}")
            
            response = conversation.invoke(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            )
            print(f"🤖 Asistente: {response.content}\n")
            
            # Obtener el historial
            history = get_session_history(session_id)
            
            # Mostrar comparación clara
            total_messages = len(history._messages)
            visible_messages = len(history.messages)
            
            print(f"📊 ESTADO DE LA MEMORIA:")
            print(f"   💾 Total almacenado: {total_messages} mensajes")
            print(f"   👁️  Visible al modelo: {visible_messages} mensajes")
            print(f"   🗑️  Mensajes descartados: {total_messages - visible_messages}")
            
            # Mensajes almacenados totalmente
            print(f"\n📚 HISTORIAL COMPLETO ALMACENADO ({total_messages} mensajes):")
            if total_messages == 0:
                print("     (Ningún mensaje aún)")
            else:
                for j, msg in enumerate(history._messages, 1):
                    role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
                    # Marcar si está en la ventana visible
                    is_visible = j > total_messages - visible_messages
                    marker = "✅" if is_visible else "❌"
                    print(f"     {j}. {marker} {role}: {content}")
            
            # Lo que ve el modelo
            print(f"\n🔍 VENTANA VISIBLE AL MODELO ({visible_messages} mensajes):")
            if visible_messages == 0:
                print("     (Ningún mensaje visible)")
            else:
                for j, msg in enumerate(history.messages, 1):
                    role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
                    print(f"     {j}. ✅ {role}: {content}")
            
            print("\n" + "="*60 + "\n")
            
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_window_memory()

=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===
Mantiene solo los 2 intercambios más recientes

==================== INTERACCIÓN 1 ====================
👤 Usuario: Mi nombre es Carlos y tengo 30 años


<ruta-local>:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


🤖 Asistente: ¡Hola, Carlos! Bienvenido. 😊 Tengo 30 años también (en el sentido de que mi "edad" como asistente virtual es de 30 años de experiencia en ayudar a las personas, aunque técnicamente no tengo una edad real). ¿En qué puedo ayudarte hoy? ¿Tienes alguna pregunta, necesitas información, consejos o simplemente quieres charlar? ¡Estoy aquí para lo que necesites! 🌟

(Por cierto, ¡30 años es una edad increíble para explorar el mundo, los sueños y los proyectos!).

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 2 mensajes
   👁️  Visible al modelo: 2 mensajes
   🗑️  Mensajes descartados: 0

📚 HISTORIAL COMPLETO ALMACENADO (2 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: ¡Hola, Carlos! Bienvenido. 😊 Tengo 30 años también (en el se...

🔍 VENTANA VISIBLE AL MODELO (2 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: ¡Hola, Carlos! Bienvenido. 😊 Tengo 30 años también (en el se...


==================== I

🤖 Asistente: ¡Qué bien, Carlos! 🎨✨ El diseño gráfico es un campo fascinante y lleno de creatividad. ¿En qué área te especializas o qué tipo de proyectos sueles desarrollar? Aquí te dejo algunas ideas para que profundicemos en lo que necesites:

### **Posibles temas en los que puedo ayudarte:**
1. **Herramientas y software:**
   - ¿Quieres aprender a dominar **Adobe Photoshop, Illustrator, InDesign, Figma, Canva o Procreate**?
   - ¿Cómo optimizar tu flujo de trabajo con **plugins, atajos de teclado o automatizaciones**?
   - Alternativas gratuitas o más económicas (como **GIMP, Krita, Affinity Designer**).

2. **Tendencias y estilo:**
   - ¿Buscas inspiración para **tipografías, paletas de color, composiciones o estilos** (minimalismo, retro, dark mode, etc.)?
   - Ejemplos de **diseños virales o campañas exitosas** para analizar.

3. **Portafolio y branding personal:**
   - ¿Cómo estructurar un **portafolio digital atractivo** (Behance, personal o web)?
   - Consejos para **diferencia

🤖 Asistente: ¡Me encanta que compartas tus pasiones, Carlos! 🎷☕ El café y el jazz son una combinación *perfecta* para inspirar creatividad (y también para sobrevivir a las largas noches de diseño). Aquí van algunas ideas que podrían interesarte, mezclando tu mundo como diseñador con tus gustos:

---

### **🎨 Café + Diseño Gráfico**
1. **Inspiración visual en el café:**
   - El **jazz y el café** tienen una estética cálida y orgánica que puede inspirar diseños. Piensa en:
     - **Paletas de color:** Tonos cálidos como *marrón chocolate, beige crema, verde musgo* o *naranja terracota* (como los de una taza de café recién hecho).
     - **Texturas:** Granos de café, vapor, gotas de agua o incluso el *grain* de los discos de vinilo (¡el jazz es *vintage* por naturaleza!).
     - **Tipografías:** Fuentes con personalidad, como *serif* elegantes (ej. **Playfair Display**) o *handwritten* (ej. **Pacifico** para un toque artesanal).
   - **Ejemplo práctico:** Diseña una **etiqueta para una ma

🤖 Asistente: ¡No tengo memoria de tus interacciones anteriores en esta conversación! 😊 Cada nueva interacción es como un "nuevo día" para mí, así que no recuerdo detalles como tu edad, gustos o proyectos específicos de antes.

Si quieres compartir algo (como tu edad, ubicación o intereses), estaré encantado de ayudarte con eso. Por ejemplo:
- Si tienes **25 años**, quizá estés en una etapa de consolidar tu carrera como diseñador.
- Si eres **más joven**, tal vez estés explorando herramientas nuevas o tu estilo personal.
- Si eres **mayor**, ¡seguro tienes una perspectiva única y mucha experiencia para compartir!

Pero no te preocupes: **no necesitas decírmelo** si no quieres. Aquí estoy para lo que necesites, ya sea:
- **Tips de diseño** (como los de jazz/café).
- **Consejos para freelancers** (facturación, clientes, etc.).
- **Inspiración** (paletas, tipografías, proyectos).
- **Algo más** (¡avísame!).

¿En qué puedo ayudarte *ahora*? 🎨☕🔥

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado

🤖 Asistente: ¡Ah, ya caigo! 😄 **Eres diseñador gráfico** (o algo relacionado con el diseño visual), según lo que mencionaste antes sobre tu trabajo y tus intereses en jazz, café y creatividad.

Si quieres confirmar o ajustar algo, dime:
- ¿Es **diseño gráfico** lo que haces?
- ¿O trabajas en algo más específico como **diseño de UI/UX, branding, ilustración, tipografía, etc.**?
- ¿O acaso estás explorando cambiar de rumbo profesional?

*(Por ejemplo, si te gusta el jazz, podrías incluso combinarlo con diseño sonoro o *motion graphics* para videos de música).*

---
### **🎨 Ideas rápidas para diseñadores (según tu perfil):**
1. **Si eres *branding*:**
   - Diseña un **logo para una cafetería de jazz** (ej: "The Velvet Bean" o "Midnight Espresso").
   - Usa **símbolos abstractos** (como un saxofón estilizado o una taza de café con líneas de música).

2. **Si te gusta el *UI/UX*:**
   - Crea un **mockup de una app de streaming de jazz** con paletas cálidas y tipografías elegantes.
   - Insp

## 3. ConversationSummaryMemory - Resumen Inteligente

Esta memoria **resume** conversaciones largas en lugar de mantener todo el texto completo, ahorrando tokens significativamente.

In [7]:

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Función para resumir automáticamente cuando hay muchos mensajes
def auto_summarize(session_id: str, max_messages=6):
    history = get_session_history(session_id)
    
    if len(history.messages) > max_messages:
        # Mensajes a resumir (todos excepto los últimos 2)
        messages_to_summarize = history.messages[:-2]
        
        # Crear texto para resumir
        conversation_text = ""
        for msg in messages_to_summarize:
            role = "Usuario" if msg.type == "human" else "Asistente"
            conversation_text += f"{role}: {msg.content}\n"
        
        # Generar resumen
        summary_response = llm.invoke(f"Resume esta conversación en 2-3 líneas:\n{conversation_text}")
        summary = summary_response.content
        
        # Reemplazar mensajes antiguos con el resumen
        recent_messages = history.messages[-2:]
        history.clear()
        history.add_ai_message(f"[RESUMEN]: {summary}")
        history.messages.extend(recent_messages)

# Crear conversación
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejemplo_summary_memory():
    print("=== CONVERSATION SUMMARY MEMORY ===")
    print("Resume conversaciones largas para ahorrar tokens\n")
    
    session_id = "summary_session"
    
    # Conversación de ejemplo
    inputs = [
        "Hola, soy María González, ingeniera de software de 35 años",
        "Trabajo en una startup de fintech en Madrid desarrollando pagos digitales",
        "Usamos React, Node.js, Docker y Kubernetes en nuestros proyectos",
        "Mi mayor desafío es la latencia en transacciones internacionales",
        "También trabajo en mejorar la UX de nuestra app móvil",
        "¿Puedes resumir quién soy y cuáles son mis principales desafíos?"
    ]
    
    try:
        for i, user_input in enumerate(inputs, 1):
            print(f"{'='*15} INTERACCIÓN {i} {'='*15}")
            print(f"👤 Usuario: {user_input}")
            
            # Resumir automáticamente si es necesario
            auto_summarize(session_id)
            
            response = conversation.invoke(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            )
            print(f"🤖 Asistente: {response.content}\n")
            
            # Mostrar estado de la memoria
            history = get_session_history(session_id)
            total_messages = len(history.messages)
            
            print(f"📊 ESTADO DE LA MEMORIA:")
            print(f"   💾 Total mensajes: {total_messages}")
            
            # Verificar si hay resumen
            has_summary = any("[RESUMEN]" in msg.content for msg in history.messages if hasattr(msg, 'content'))
            print(f"   📝 Tiene resumen: {'✅ Sí' if has_summary else '❌ No'}")
            
            print(f"\n💬 CONTENIDO ACTUAL DE LA MEMORIA:")
            for j, msg in enumerate(history.messages, 1):
                role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                content = msg.content
                
                # Destacar si es un resumen
                if "[RESUMEN]" in content:
                    role = "📝 Resumen"
                    content = content.replace("[RESUMEN]: ", "")
                
                # Truncar si es muy largo
                if len(content) > 80:
                    content = content[:80] + "..."
                
                print(f"   {j}. {role}: {content}")
            
            print("\n" + "="*50 + "\n")
            
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_summary_memory()

=== CONVERSATION SUMMARY MEMORY ===
Resume conversaciones largas para ahorrar tokens

=============== INTERACCIÓN 1 ===============
👤 Usuario: Hola, soy María González, ingeniera de software de 35 años


<ruta-local>:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


🤖 Asistente: ¡Hola, María! Encantado de conocerte. 😊 Como ingeniera de software con esa experiencia y perspectiva, seguro tienes un perfil muy interesante. ¿En qué puedo ayudarte hoy? Aquí van algunas ideas según lo que podría interesarte (o dime tú mismo qué necesitas):

1. **Desarrollo profesional**:
   - ¿Buscas consejos para **crecer en tu carrera** (ej.: liderazgo técnico, mentorías, certificaciones)?
   - ¿Te interesa **transicionar a áreas emergentes** como IA, DevOps, o cloud computing?
   - ¿Necesitas ayuda con **entrevistas técnicas** o cómo destacar en un proceso de selección?

2. **Tecnologías y herramientas**:
   - ¿Quieres profundizar en algún **lenguaje/framework** (ej.: Rust, Go, Kubernetes, TypeScript)?
   - ¿Buscas recomendaciones de **libros, cursos o comunidades** (como Women Who Code, DevOps Days)?
   - ¿Cómo optimizar **buenas prácticas de código** o arquitectura de software?

3. **Equilibrio vida-trabajo**:
   - ¿Cómo manejas el **burnout** o la conciliación fami

🤖 Asistente: ¡Qué emocionante! Trabajar en una **startup de fintech en Madrid** es un entorno dinámico, lleno de desafíos técnicos y oportunidades para innovar en pagos digitales. 💡💳 Aquí tienes algunas ideas personalizadas según tu contexto, pero **dime si alguna de estas áreas te interesa profundizar o si prefieres otro enfoque**:

---

### **1. Desafíos técnicos en pagos digitales (y cómo abordarlos)**
Madrid es un hub de fintech con regulaciones como **PSD2, SCA (Strong Customer Authentication)** y normativas de **fraude y cumplimiento (AML/KYC)**. Algunos temas clave:
- **Integración con APIs bancarias**:
  - ¿Estás usando **Open Banking** (ej.: APIs de bancos españoles como CaixaBank, BBVA) o soluciones como **Tuenti Pay, Bizum, o Stripe**?
  - *Ejemplo práctico*: ¿Cómo manejas los **3DS (3-D Secure)** para autenticación en pagos online?
- **Escalabilidad y latencia**:
  - Los pagos en tiempo real requieren **bajas latencias** y sistemas distribuidos. ¿Usas **Kafka, RabbitMQ, o m

🤖 Asistente: ¡Perfecto! Con **React, Node.js, Docker y Kubernetes** en tu stack, estás en un entorno moderno y escalable, ideal para una **startup de fintech** donde la **latencia, seguridad y escalabilidad** son críticas en pagos digitales. Vamos a profundizar en cómo optimizar estos componentes para tu contexto, con ejemplos prácticos y desafíos típicos de fintech.

---

### **1. Arquitectura para pagos digitales (con tu stack)**
#### **Frontend: React**
- **Pagos en tiempo real**:
  - Usa **React Hooks** (como `useEffect` para eventos en tiempo real) o **WebSockets** (con librerías como `socket.io`) para notificaciones instantáneas de transacciones.
  - *Ejemplo*: Mostrar el estado de un pago (pendiente/confirmado/rechazado) en tiempo real.
  - **Seguridad**:
    - Implementa **CSRF tokens** y **CORS estrictos** para evitar ataques en APIs de pagos.
    - Usa **React Query** o **SWR** para manejar estados de carga/error de forma eficiente.

- **Integración con APIs de pagos**:
  - U

🤖 Asistente: ¡Entiendo perfectamente el desafío! **La latencia en transacciones internacionales** es uno de los mayores dolores de cabeza en fintech, especialmente cuando trabajas con pagos digitales en una startup. Los usuarios esperan **instantaneidad** (o casi), pero los factores como:
- **Diferencias horarias y zonas geográficas** (ej.: pagar desde Latinoamérica a Europa).
- **Dependencia de bancos tradicionales** (que pueden tardar horas/días en procesar).
- **Límites de velocidad en APIs de pagos** (ej.: Stripe, Bizum, o bancos locales).
- **Redes de comunicación inestables** (latencia en DNS, conexiones satelitales, etc.).
- **Fronteras regulatorias** (ej.: PSD2 en Europa vs. normativas locales en otros países).

---
### **Soluciones técnicas para reducir latencia (con tu stack: React + Node.js + Docker + Kubernetes)**
Vamos a dividir el problema en **capas** (frontend, backend, infraestructura) y darte soluciones prácticas, con ejemplos concretos para tu contexto.

---

## **1.

🤖 Asistente: ¡Perfecto! Mejorar la **UX (Experiencia de Usuario) en la app móvil** es clave para retener usuarios, reducir la tasa de abandono y aumentar la conversión en pagos internacionales. Dado que trabajas con **React (frontend) + Node.js (backend) + fintech**, vamos a enfocarnos en soluciones **técnicas, psicológicas y de diseño** que puedas implementar rápidamente, priorizando lo que más impacto tendrá en tu contexto.

---

## **📌 Desafíos comunes en UX de apps de fintech (y cómo solucionarlos)**
### **1. Problema: "El usuario no entiende cómo funciona el pago internacional"**
**Ejemplo**: *"¿Por qué mi transferencia a Argentina tarda 3 días?"* o *"¿Por qué me cobran más?"*.
**Soluciones**:
#### **A. Microcopias claras y educativas**
- **Evita jerga técnica**: Usa lenguaje sencillo.
  - ❌ *"Procesamiento en tiempo real con SCA 3DS 2.0"*
  - ✅ *"Pago seguro en segundos (verificamos tu tarjeta con un código)"*.
- **Explica los tiempos de espera**:
  ```jsx
  // Ejemplo en React N

🤖 Asistente: **[RESUMEN EXPRESIVO]**

**👩‍💻 QUIÉN ERES:**
- **Desarrolladora Full-Stack** en una *startup de fintech en Madrid* (React + Node.js + Docker/Kubernetes).
- Especializada en **pagos digitales internacionales**, con enfoque en **escalabilidad, seguridad y cumplimiento regulatorio** (ej.: SCA 3DS 2.0, PSD2).
- Trabajas en un entorno técnico complejo donde la **latencia en transacciones** y la **experiencia de usuario (UX)** son críticos para retener clientes.

---

**🔥 DESAFÍOS PRINCIPALES (priorizados):**
1. **📉 Latencia en pagos internacionales**
   - Transacciones lentas (horas/días) por dependencia de bancos tradicionales o APIs externas (ej.: Stripe, Wise).
   - **Impacto**: Usuarios abandonan el checkout o pierden confianza.
   - **Tecnologías afectadas**: Backend (Node.js), infraestructura (Kubernetes), APIs de pagos.

2. **😤 UX móvil frustrante**
   - Flujos de pago **largos y confuso**s (ej.: redirecciones a 3DS 2.0, falta de feedback claro).
   - **Impacto**: Alta t

## Consideraciones Técnicas y Mejores Prácticas

### Selección del Tipo de Memoria

| Tipo | Cuándo Usarlo | Ventajas | Desventajas |
|------|---------------|----------|-------------|
| **Buffer** | Conversaciones cortas | Contexto completo | Alto consumo de tokens |
| **Window** | Contexto reciente importante | Eficiente en tokens | Puede perder información clave |
| **Summary** | Conversaciones muy largas | Balance eficiencia/contexto | Pérdida de detalles específicos |

### Mejores Prácticas:

1. **Gestión de Tokens**:
   - Monitorea el uso de tokens regularmente
   - Establece límites máximos para evitar costos excesivos
   - Considera el costo vs. calidad del contexto

2. **Selección Estratégica**:
   - Usa Buffer para sesiones cortas e importantes
   - Usa Window para conversaciones con contexto limitado
   - Usa Summary para sesiones largas de asistencia

3. **Optimización**:
   - Limpia memoria periódicamente si es necesario
   - Implementa estrategias híbridas según el caso de uso
   - Considera almacenamiento persistente para memoria a largo plazo

## Ejercicios Prácticos

### Ejercicio 1: Análisis de Consumo
Implementa un sistema que monitoree y reporte el uso de tokens con diferentes tipos de memoria.

### Ejercicio 2: Memoria Híbrida
Diseña una estrategia que combine multiple tipos de memoria según el contexto.

### Ejercicio 3: Persistencia
Extiende el chatbot para guardar y cargar memoria entre sesiones.

## Conceptos Clave Aprendidos

1. **Importancia de la memoria** en conversaciones naturales
2. **Tipos de memoria** y sus casos de uso específicos
3. **Balance** entre contexto y eficiencia de tokens
4. **Implementación práctica** con LangChain
5. **Estrategias de optimización** para diferentes escenarios

## Conclusión del Módulo IL1.1

Has completado la introducción a LLMs y conexiones API. Los conceptos aprendidos:

1. **APIs directas** vs **frameworks** como LangChain
2. **Streaming** para mejor experiencia de usuario
3. **Memoria** para conversaciones contextuales
4. **Mejores prácticas** de seguridad y optimización

### Próximos Pasos
En **IL1.2** exploraremos técnicas avanzadas de **prompt engineering** incluyendo zero-shot, few-shot, y chain-of-thought prompting.